# Spike Sparsity and Correlation

Cumulative distribution of shift sparsity and Delta vs BA.2 shift correlations.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import multidms

from notebooks._common import load_config, combine_replicate_muts

In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]
output_dir = spike["output_dir"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
times_seen_threshold = spike["times_seen_threshold"]

warnings.simplefilter("ignore")
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))
mutations_df = pd.read_csv(f"{output_dir}/mutations_df.csv", index_col=0)
mc = multidms.ModelCollection(models.drop(columns="replicate", errors="ignore"))
print(f"Loaded {len(models)} models, {len(mutations_df)} mutations")

## Cumulative distribution of sparsity

In [ ]:
tall_mut_df_chosen = (
    mc.split_apply_combine_muts(
        query=f"fusionreg == {chosen_lasso_strength}",
        times_seen_threshold=times_seen_threshold,
    )
    .reset_index()
    .rename(columns={"dataset_name": "replicate"})
    .assign(sense=lambda x: ["stop" if "*" in mut else "nonsynonymous" for mut in x.mutation])
    .melt(
        id_vars=["replicate", "mutation", "sense"],
        value_vars=["shift_Delta", "shift_Omicron_BA2"],
        var_name="condition",
        value_name="shift",
    )
    .replace({"shift_Delta": "Delta", "shift_Omicron_BA2": "Omicron_BA2"})
)

saveas = "percent_shifts_under_x_lineplot"
fig, ax = plt.subplots(1, 2, figsize=[6.4, 3.5], sharey="row")

for col, (condition, cdf) in enumerate(tall_mut_df_chosen.groupby("condition")):
    iter_ax = ax[col]
    for sense, sdf in cdf.groupby("sense"):
        abs_shifts = sdf["shift"].abs().sort_values()
        cdf_vals = np.arange(1, len(abs_shifts) + 1) / len(abs_shifts) * 100
        iter_ax.plot(abs_shifts.values, cdf_vals, label=sense, linewidth=2)
    iter_ax.set_xlabel("|shift|")
    iter_ax.set_ylabel("% mutations with |shift| ≤ x" if col == 0 else "")
    iter_ax.set_title(condition.replace("Omicron_", ""))
    iter_ax.legend()
    sns.despine(ax=iter_ax)

plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()

## Delta vs BA.2 shift correlation

In [ ]:
mut_df_replicates = (
    combine_replicate_muts(
        {
            f"{fit.dataset_name}".split("-")[-1]: fit.model
            for fit in models.query(f"fusionreg == {chosen_lasso_strength}").itertuples()
        },
        times_seen_threshold=times_seen_threshold,
        how="inner",
    )
    .assign(sense=lambda x: ["stop" if "*" in mut else "nonsynonymous" for mut in x.muts])
)

saveas = "shift_corr_Delta_BA2"
fig, ax = plt.subplots(1, figsize=[4, 4])

data = mut_df_replicates.dropna()
x = data["avg_shift_Delta"]
y = data["avg_shift_Omicron_BA2"]

for sense, color in [("nonsynonymous", "grey"), ("stop", "red")]:
    sdf = data.query(f"sense == '{sense}'")
    ax.scatter(sdf["avg_shift_Delta"], sdf["avg_shift_Omicron_BA2"], alpha=0.4, s=10, c=color, label=sense)

ax.plot([-3, 3], [-3, 3], "--", c="royalblue", lw=1)
corr = pearsonr(x, y)[0] ** 2
ax.annotate(f"$R^2 = {corr:.2f}$", (0.05, 0.9), xycoords="axes fraction", fontsize=12)
ax.set_xlabel("shift (Delta)")
ax.set_ylabel("shift (BA.2)")
ax.legend()
sns.despine()
plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()